# Anarkali: int8 export (CPU only)

Adds an int8 ONNX graph to a finished run, without retraining and without a GPU.
It downloads `RUN_NAME/web-run/best.pt` from HF, runs `scripts/export_onnx.py` with the int8
recipes, and uploads the result to `RUN_NAME/release-web/`. The fp32 files already there are kept.

Kaggle: Accelerator **None** (CPU), Internet **On**, secret **HF_TOKEN** attached.
Then Save Version → Save & Run All (Commit). It takes about 1 hour.


In [ ]:
RUN_NAME = "run2"
HF_REPO = "toufiqqureshi651/anarkali-web"
REPO_URL = "https://github.com/ToufiqQureshi/anarkali.git"
BRANCH = "main"
MAX_INT8_DRIFT = 0.02   # an int8 graph ships only if no probability moves more than this and no answer flips

In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO = WORK / "anarkali"
LOG = WORK / "int8-export-log.txt"

def sh(cmd):
    """Run a shell command, streaming its output to the notebook and LOG; raise if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=str(REPO) if REPO.exists() else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
                            bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    with open(LOG, "a", encoding="utf-8") as log:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, cmd)

if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))   # the kernel started before pip install -e
sh('pip install -q -e ".[train,onnx]" onnx')

def _hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        raise RuntimeError("attach the HF_TOKEN secret (Add-ons -> Secrets) and run again")
    return token

from huggingface_hub import HfApi, snapshot_download
HF_TOKEN = _hf_token()
hf = HfApi(token=HF_TOKEN)
print("Hugging Face user:", hf.whoami()["name"])
print("CPUs:", os.cpu_count())

In [ ]:
# best.pt, its test report, and the development + calibration splits (parity rows and int8 calibration).
got = Path(snapshot_download(HF_REPO, token=HF_TOKEN, allow_patterns=[
    f"{RUN_NAME}/web-run/best.pt", f"{RUN_NAME}/web-run/test-report.json",
    f"{RUN_NAME}/web-decisions-v0/development.jsonl", f"{RUN_NAME}/web-decisions-v0/calibration.jsonl",
    f"{RUN_NAME}/web-decisions-v0/manifest.json"]))
CKPT = got / RUN_NAME / "web-run" / "best.pt"
DATA = got / RUN_NAME / "web-decisions-v0"
report = json.loads((got / RUN_NAME / "web-run" / "test-report.json").read_text())
temps = json.dumps(report["temperature_by_type"])
print("checkpoint:", CKPT, f"{CKPT.stat().st_size / 1e6:.0f} MB")
print("temperature by type:", temps)

In [ ]:
OUT = WORK / "release-int8"
try:
    sh(f"python scripts/export_onnx.py --checkpoint {CKPT} --output {OUT} --data {DATA} "
       f"--name 'Anarkali Web 150M' --temperature-by-type '{temps}' --max-int8-drift {MAX_INT8_DRIFT}")
finally:
    # upload whatever was measured, even if a later step failed
    for name in ("model.int8.onnx", "parity.json", "anarkali.json"):
        if (OUT / name).exists():
            hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(OUT / name),
                           path_in_repo=f"{RUN_NAME}/release-web/{name}")
            print("uploaded", f"{RUN_NAME}/release-web/{name}")
    hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(LOG), path_in_repo=f"{RUN_NAME}/int8-export-log.txt")

In [ ]:
parity = json.loads((OUT / "parity.json").read_text())
print(f"{'graph':32s} {'passed':>6s} {'drift':>8s} {'flips':>5s} {'p50 ms':>7s}")
for name, m in parity.items():
    if name in ("thread_latency", "selected_int8") or not isinstance(m, dict):
        continue
    if "error" in m:
        print(f"{name:32s} {'no':>6s}  error: {m['error'][:80]}")
        continue
    print(f"{name:32s} {str(m['passed']):>6s} {m['max_probability_drift']:8.4f} {m['argmax_mismatches']:5d} "
          f"{m['cpu_latency_p50_ms']:7.0f}")
best = parity.get("selected_int8")
print()
print("shipped int8:", best["recipe"] if best else "none (no recipe passed the parity gate)")
for graph, by_threads in parity["thread_latency"].items():
    print(graph, {t: round(v["cpu_latency_p50_ms"]) for t, v in by_threads.items()})